In [ ]:
import pandas as pd
import json

X = pd.read_parquet("/context/data/X.parquet")
y = pd.read_parquet("/context/data/y.parquet")

with open("/context/data/moons_split.json") as f:
    splits = json.load(f)

X_train = X[X["moon"].isin(splits["train"])]
y_train = y[y["moon"].isin(splits["train"])]

X_test_cloud = X[X["moon"].isin(splits["reduced_cloud"])]
y_test_cloud = y[y["moon"].isin(splits["reduced_cloud"])]


In [ ]:
import joblib
import os
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from scipy.stats import rankdata


def train(X_train, y_train, model_directory_path, loop_moon, embargo):
    # ── All config lives here (no outer-scope dependencies) ───────────────────
    feature_cols = [c for c in X_train.columns if c.startswith("Feature_")]

    # ── Merge and drop missing targets ────────────────────────────────────────
    df = X_train.merge(y_train[["moon", "id", "target"]], on=["moon", "id"])
    df = df.dropna(subset=["target"])

    # ── Fill NaN cross-sectionally per moon (NOT global median) ──────────────
    # Global median leaks info across moons and distorts cross-sectional ranking
    df[feature_cols] = df.groupby("moon")[feature_cols].transform(
        lambda s: s.fillna(s.median())
    )
    df[feature_cols] = df[feature_cols].fillna(0.0)

    # ── Rank-normalise target cross-sectionally per moon ──────────────────────
    # Stabilises training: each moon contributes equally regardless of scale
    def cs_rank(s):
        n = len(s)
        if n < 2: return s * 0.0
        r = s.rank(method="average")
        return (r - 1) / (n - 1) - 0.5

    df["target_rank"] = df.groupby("moon")["target"].transform(cs_rank)

    X_fit = df[feature_cols]
    y_fit = df["target_rank"]

    print(f"  Training on {X_train['moon'].nunique()} moons, "
          f"{len(df):,} rows, {len(feature_cols)} features")

    model = LGBMRegressor(
        n_estimators=250,
        learning_rate=0.04,
        num_leaves=128,
        subsample=0.75,
        colsample_bytree=0.7,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )
    model.fit(X_fit, y_fit)

    # Save model + feature list together
    joblib.dump(
        {"model": model, "features": feature_cols},
        os.path.join(model_directory_path, "model.joblib"),
    )
    print("  Model saved.")


In [ ]:
def infer(X_test, model_directory_path, loop_moon, embargo):

    obj      = joblib.load(os.path.join(model_directory_path, "model.joblib"))
    model    = obj["model"]
    features = obj["features"]

    X = X_test.copy()

    # Fill NaN with cross-sectional median for this moon
    for c in features:
        if c in X.columns:
            med = X[c].median()
            X[c] = X[c].fillna(med if not np.isnan(med) else 0.0)

    # Predict
    raw_preds = model.predict(X[features])

    # Rank-normalise predictions to [-0.5, 0.5]
    # This is what the scorer compares — makes scale consistent across moons
    n = len(raw_preds)
    final = (rankdata(raw_preds) - 1) / max(n - 1, 1) - 0.5

    return pd.DataFrame({
        "moon":       X_test["moon"].values,
        "id":         X_test["id"].values,
        "prediction": final,
    })


In [ ]:
from scipy.stats import pearsonr
import os

embargo = 4
model_dir = "./model_internet"
os.makedirs(model_dir, exist_ok=True)

train(X_train, y_train, model_dir, loop_moon=0, embargo=embargo)

results = []
for moon in splits["reduced_cloud"]:
    X_moon = X_test_cloud[X_test_cloud["moon"] == moon]
    pred   = infer(X_moon, model_dir, loop_moon=moon, embargo=embargo)
    results.append(pred)

predictions = pd.concat(results)

for moon in splits["reduced_cloud"]:
    p      = predictions[predictions["moon"] == moon]
    gt     = y_test_cloud[y_test_cloud["moon"] == moon]
    merged = p.merge(gt, on=["moon", "id"])
    if len(merged) > 1:
        corr, _ = pearsonr(merged["prediction"], merged["target"])
        print(f"Moon {moon}: Pearson r = {corr:.4f}")
